In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage , SystemMessage
import requests
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("CURRENCY_API_KEY")
BASE_URL = os.getenv("CURRENCY_API_URL")

In [2]:
# tool Creation
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str,target_currency:str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency"""
    
    url = f"{BASE_URL}/{API_KEY}/pair/{base_currency}/{target_currency}"
    
    response = requests.get(url)
    
    return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])-> float:
    """Given currency conversion rate this function calculate the target currency value from a given base currency value"""
    
    return base_currency_value * conversion_rate
    

In [3]:
convert.invoke({'base_currency_value':10,'conversion_rate':96.2365})

962.365

In [4]:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [5]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [7]:
human_input = HumanMessage('convert 9 USD to INR')

In [14]:
messages = []
messages.append(human_input)

In [15]:
ai_message1 = llm_with_tools.invoke(messages)

In [16]:
ai_message1.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'ece7d7e8-29d0-4396-9672-2805ea41314a',
  'type': 'tool_call'}]

In [17]:
messages.append(ai_message1)

In [21]:
import json
for tool_call in ai_message1.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        mess_1 = get_conversion_factor.invoke(tool_call)

        conversion_rate = json.loads(mess_1.content)['conversion_rate']
        messages.append(ai_message1)
        messages.append(mess_1)
        
        ai_message2 = llm_with_tools.invoke(messages)
        
        print(ai_message2.content)
    

In [22]:
import json
for tool_call in ai_message1.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        mess_1 = get_conversion_factor.invoke(tool_call)

        conversion_rate = json.loads(mess_1.content)['conversion_rate']
        messages.append(mess_1)
        
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        
        mess_2 = convert.invoke(tool_call)
        messages.append(mess_2)